# 10 — Corrected feature extraction and head refit

Implements `notebook10_spec.md`. **Only setup and Section 1 are written.** Sections 2–7
(unpadded re-extraction, level, dedup, negatives, refit, write-back) are gated on Section
1's numbers and are added after they have been reviewed. Nothing is re-extracted here.

## Section 1 — padding diagnostic

Notebook 04's `degrade()` filled the 4 s window's padding with noise; `X_med` and live
inference pad with zeros. VIVAE positives are ~1 s and CREMA-D negatives ~2.5 s, so
padding amount and content correlate with the label. This section holds audio content
constant and varies only the padding:

| condition | what WavLM sees |
|---|---|
| `zeros` | clip, then zero padding to 4 s (the pipeline and `X_med` today) |
| `noise` | `degrade(clip + zero pad)`, verbatim notebook 04 (what the heads were trained on) |
| `noise_padonly` | clip untouched, Gaussian noise only in the pad region (padding isolated) |
| `none` | clip at its own length (crop >4 s from the start), mean-pooled over its real frames |

`noise` is the spec's condition and reproduces training, but `degrade()` also low-passes
and adds noise *over the clip*, so it changes content as well as padding.
`noise_padonly` is added so the padding effect can be read on its own.

Readouts:
1. **Padding-type AUC**: logistic regression predicting the padding condition from the
   embedding, GroupKFold on clip id (the same clip never sits on both sides).
2. **VIVAE-vs-CREMA-D AUC within each condition**, speaker-grouped like the original gate.
   `none` is the honest estimate; compare it with the published 0.996.
3. Duration distributions and Spearman(duration, current gate score) per corpus.

All audio goes through `src.preprocess.load_audio` (−23 LUFS; spec §3). The padded
conditions are embedded with `src.pipeline.embed`.

**Inputs (Kaggle):** `ejlok1/cremad`; a private dataset built from this repo
(`src/`, `models/*.joblib`, `notebooks/04_acoustic_branch.ipynb`); internet on for
VIVAE (Zenodo). GPU recommended.

In [ ]:
import os, sys, re, glob, json, time, random, hashlib, zipfile, urllib.request
from pathlib import Path

ON_KAGGLE = Path("/kaggle/input").exists()

def find_repo():
    cands = [Path.cwd(), Path.cwd().parent]
    if ON_KAGGLE:
        cands += [p.parent.parent for p in Path("/kaggle/input").glob("**/src/pipeline.py")]
    for c in cands:
        if (c / "src" / "pipeline.py").exists():
            return c.resolve()
    raise FileNotFoundError("src/pipeline.py not found; attach the code dataset")

REPO = find_repo()
sys.path.insert(0, str(REPO))

CONFIG = {
    "seed": 42,
    "n_per_corpus": int(os.environ.get("CV10_N", 200)),
    "crema_dir": os.environ.get("CV10_CREMA_DIR", "/kaggle/input"),
    "vivae_dir": os.environ.get("CV10_VIVAE_DIR", "/kaggle/working/vivae"),
    "vivae_url": "https://zenodo.org/records/4066235/files/VIVAE.zip?download=1",
    "vivae_emotions": ["pain", "fear"],       # the gate's positives
    "crema_emotion": "NEU",                   # the gate's calm-speech negatives
    "out_dir": Path(os.environ.get("CV10_OUT",
                    "/kaggle/working/results" if ON_KAGGLE else str(REPO / "results"))),
    "cv_folds": 5,
    "published_auc_vs_calm": 0.996,           # results/distress_gate_results.json
}
CONFIG["out_dir"].mkdir(parents=True, exist_ok=True)
OUT = CONFIG["out_dir"]

LOCAL_WAVLM = REPO / "models" / "wavlm-base-plus"
os.environ["HF_HUB_OFFLINE"] = "1" if LOCAL_WAVLM.exists() else "0"

import numpy as np, pandas as pd, torch, joblib, soundfile as sf
import matplotlib.pyplot as plt
from scipy.signal import butter, lfilter
from scipy.stats import spearmanr
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import roc_auc_score
from transformers import AutoFeatureExtractor, WavLMModel

from src.preprocess import load_audio, pad_or_crop, stable_seed
from src.pipeline import CONFIG as PIPE, embed
from src.rules import CONFIG as RULES

random.seed(CONFIG["seed"]); np.random.seed(CONFIG["seed"]); torch.manual_seed(CONFIG["seed"])
SR = PIPE["sr"]
WIN = int(round(PIPE["acoustic_window_s"] * SR))
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print(f"repo={REPO} on_kaggle={ON_KAGGLE} device={DEV} seed={CONFIG['seed']} window={WIN/SR}s")

In [ ]:
# ---- models: frozen WavLM in fp32 (as the pipeline runs it) and the CURRENT gate ----
src_wavlm = str(LOCAL_WAVLM) if LOCAL_WAVLM.exists() else "microsoft/wavlm-base-plus"
fe = AutoFeatureExtractor.from_pretrained(src_wavlm)
wavlm = WavLMModel.from_pretrained(src_wavlm).to(DEV).eval()
for p in wavlm.parameters():
    p.requires_grad = False
print("feat_extract_norm:", wavlm.config.feat_extract_norm, "| do_normalize:", fe.do_normalize)

gate_b = joblib.load(REPO / "models" / "distress_gate.joblib")
GATE, POS = gate_b["model"], list(gate_b["model"].classes_).index(1)
gate_score = lambda X: GATE.predict_proba(np.atleast_2d(X))[:, POS]

@torch.no_grad()
def emb_native(wav):
    # Batch size 1 at the clip's own length: every frame is real, so the mean pool is
    # over real frames only. No attention mask needed (and none passed: group-norm
    # WavLM was pretrained without one).
    x = fe(wav.astype(np.float32), sampling_rate=SR, return_tensors="pt").input_values.to(DEV)
    return wavlm(x).last_hidden_state.mean(1).float().cpu().numpy()[0]

def emb_window(wav):
    return embed(fe, wavlm, wav, DEV)[0]

In [ ]:
def md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

# ---- VIVAE: same discovery as notebook 04, then dedup by CONTENT hash (spec §4) ----
DST = CONFIG["vivae_dir"]
if not glob.glob(f"{DST}/**/*.wav", recursive=True):
    os.makedirs(DST, exist_ok=True)
    urllib.request.urlretrieve(CONFIG["vivae_url"], f"{DST}.zip")
    with zipfile.ZipFile(f"{DST}.zip") as z:
        z.extractall(DST)
vv = sorted(glob.glob(f"{DST}/**/*.wav", recursive=True))
rows = []
for i, p in enumerate(vv):
    spk, emo, inten, item = os.path.splitext(os.path.basename(p))[0].split("_")
    rows.append({"nb04_index": i, "path": p, "file": os.path.basename(p), "subset": Path(p).parent.name,
                 "speaker": spk, "emotion": emo, "intensity": inten, "md5": md5(p),
                 "duration_s": sf.info(p).duration})
vd_all = pd.DataFrame(rows)
g = vd_all.groupby("md5")
dup_groups = g.filter(lambda x: len(x) > 1)
print(f"VIVAE files {len(vd_all)} -> unique by content {vd_all.md5.nunique()}")
print(f"  content-duplicate groups: {dup_groups.md5.nunique()}, "
      f"of which with differing basenames: {int((dup_groups.groupby('md5').file.nunique() > 1).sum())}")
print(vd_all.groupby("subset").size().to_string())
vd = vd_all.drop_duplicates("md5").reset_index(drop=True)
vd.drop(columns=["path"]).to_csv(OUT / "padding_diagnostic_vivae_index.csv", index=False)

# ---- CREMA-D ----
EMO = {"ANG": "anger", "DIS": "disgust", "FEA": "fear", "HAP": "happy", "NEU": "neutral", "SAD": "sad"}
crows = []
for p in sorted(glob.glob(f"{CONFIG['crema_dir']}/**/*.wav", recursive=True)):
    if "crema" not in p.lower() and "audiowav" not in p.lower():
        continue
    m = re.match(r"(\d+)_([A-Z]+)_([A-Z]{3})_([A-Z]{2})\.wav$", os.path.basename(p))
    if m:
        crows.append({"path": p, "file": os.path.basename(p), "speaker": f"crema_{m.group(1)}",
                      "emo_code": m.group(3), "int_code": m.group(4)})
cd = pd.DataFrame(crows).drop_duplicates("file").reset_index(drop=True)
print(f"\nCREMA-D clips {len(cd)}; {CONFIG['crema_emotion']}: {(cd.emo_code == CONFIG['crema_emotion']).sum()}")

# ---- the fixed clip set ----
n = CONFIG["n_per_corpus"]
vsel = vd[vd.emotion.isin(CONFIG["vivae_emotions"])]
vsel = vsel.sample(n=min(n, len(vsel)), random_state=CONFIG["seed"])
csel = cd[cd.emo_code == CONFIG["crema_emotion"]]
csel = csel.sample(n=min(n, len(csel)), random_state=CONFIG["seed"])
clips = pd.concat([
    pd.DataFrame({"clip_id": "vivae/" + vsel.md5, "corpus": "VIVAE", "path": vsel.path,
                  "speaker": "vivae_" + vsel.speaker, "label": 1}),
    pd.DataFrame({"clip_id": "crema/" + csel.file, "corpus": "CREMA-D", "path": csel.path,
                  "speaker": csel.speaker, "label": 0}),
]).reset_index(drop=True)
print(f"\nclip set: {clips.corpus.value_counts().to_dict()}, speakers {clips.speaker.nunique()}")

In [ ]:
# ---- notebook 04, cell 3: degrade (verbatim) ----
def degrade(wav, sr=16000, seed=0):
    r = np.random.default_rng(seed)
    cutoff = r.uniform(3000, 7000)
    b, a = butter(6, cutoff / (sr/2), btype="low")
    w = lfilter(b, a, wav)
    snr = r.uniform(5, 25)
    sp = np.mean(w**2)
    if sp > 1e-10:
        w = w + r.normal(0, np.sqrt(sp / (10**(snr/10))), len(w))
    w = w * r.uniform(0.5, 1.5)
    peak = np.abs(w).max()
    if peak > 0.99:
        w = w * (0.99 / peak)
    return w.astype(np.float32)

def noise_padonly(clip, seed):
    # Pad region gets Gaussian noise at degrade()'s level (same SNR draw, power measured
    # over the padded window as degrade does); the clip samples are left untouched.
    w = pad_or_crop(clip, WIN).astype(np.float64)
    n_real = min(len(clip), WIN)
    r = np.random.default_rng(seed)
    r.uniform(3000, 7000)                     # consume the cutoff draw to keep SNR aligned
    snr = r.uniform(5, 25)
    sp = np.mean(w**2)
    if sp > 1e-10 and n_real < WIN:
        w[n_real:] = r.normal(0, np.sqrt(sp / (10**(snr/10))), WIN - n_real)
    return w.astype(np.float32)

CONDITIONS = {
    "zeros":         lambda clip, s: emb_window(clip),
    "noise":         lambda clip, s: emb_window(degrade(pad_or_crop(clip, WIN), seed=s)),
    "noise_padonly": lambda clip, s: emb_window(noise_padonly(clip, s)),
    "none":          lambda clip, s: emb_native(clip[:WIN]),
}

feats = {c: [] for c in CONDITIONS}
durs, t0 = [], time.time()
for k, r in enumerate(clips.itertuples()):
    clip = load_audio(r.path, target_sr=SR)          # spec §3: the one preprocessing function
    durs.append(len(clip) / SR)
    s = stable_seed(r.clip_id, "padding_diagnostic")
    for c, fn in CONDITIONS.items():
        feats[c].append(fn(clip, s))
    if (k + 1) % 50 == 0 or k + 1 == len(clips):
        print(f"  {k+1}/{len(clips)}  {time.time()-t0:.0f}s")
clips["duration_s"] = durs
feats = {c: np.vstack(v) for c, v in feats.items()}
for c, X in feats.items():
    np.save(OUT / f"padding_diagnostic_X_{c}.npy", X)
clips.drop(columns=["path"]).to_csv(OUT / "padding_diagnostic_clips.csv", index=False)
print("saved features:", {c: X.shape for c, X in feats.items()})

In [ ]:
def lr():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, class_weight="balanced"))

def oof_auc(X, y, groups):
    k = min(CONFIG["cv_folds"], len(np.unique(groups)))
    p = cross_val_predict(lr(), X, y, cv=GroupKFold(k), groups=groups, method="predict_proba")[:, 1]
    return float(roc_auc_score(y, p))

# 1. padding type, clip identity held out
pad_auc = {}
for a, b in [("zeros", "noise"), ("zeros", "noise_padonly"), ("zeros", "none"),
             ("noise", "none"), ("noise_padonly", "none")]:
    X = np.vstack([feats[a], feats[b]])
    y = np.r_[np.zeros(len(clips)), np.ones(len(clips))]
    g = np.r_[clips.clip_id.values, clips.clip_id.values]
    pad_auc[f"{a}_vs_{b}"] = oof_auc(X, y, g)

# 2. VIVAE vs CREMA-D within each condition, speaker-grouped (as the original gate)
corpus_auc = {c: oof_auc(feats[c], clips.label.values, clips.speaker.values) for c in CONDITIONS}

print("PADDING-TYPE AUC (clip-grouped):")
for k, v in pad_auc.items():
    print(f"  {k:28s} {v:.3f}")
print(f"\nVIVAE pain/fear vs CREMA-D neutral AUC (speaker-grouped); published: {CONFIG['published_auc_vs_calm']}")
for k, v in corpus_auc.items():
    print(f"  {k:14s} {v:.3f}")

In [ ]:
# 3. durations and Spearman(duration, CURRENT gate score)
for c in ("zeros", "noise", "none"):
    clips[f"gate_{c}"] = gate_score(feats[c])

print("duration (s) by corpus:")
print(clips.groupby("corpus").duration_s.describe(percentiles=[.05, .5, .95]).round(2).to_string())
print(f"fraction shorter than the {WIN/SR:.0f} s window: "
      f"{clips.assign(s=clips.duration_s < WIN/SR).groupby('corpus').s.mean().round(3).to_dict()}")

dur_rows = []
for corpus, sub in clips.groupby("corpus"):
    for c in ("zeros", "noise", "none"):
        rho, p = spearmanr(sub.duration_s, sub[f"gate_{c}"])
        dur_rows.append({"corpus": corpus, "condition": c, "n": len(sub), "spearman_rho": float(rho), "p": float(p),
                         "median_gate": float(sub[f"gate_{c}"].median()),
                         "frac_above_DISTRESS_HIGH": float((sub[f"gate_{c}"] >= RULES["DISTRESS_HIGH"]).mean())})
dur_tbl = pd.DataFrame(dur_rows)
print("\nSpearman(duration, current distress-gate score):")
print(dur_tbl.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
clips.drop(columns=["path"]).to_csv(OUT / "padding_diagnostic_clips.csv", index=False)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
names = list(corpus_auc) + [""] + list(pad_auc)
vals = list(corpus_auc.values()) + [np.nan] + list(pad_auc.values())
cols = ["tab:red"] * len(corpus_auc) + ["w"] + ["tab:gray"] * len(pad_auc)
ax[0].barh(range(len(vals)), vals, color=cols)
ax[0].set_yticks(range(len(vals))); ax[0].set_yticklabels(names, fontsize=8); ax[0].invert_yaxis()
ax[0].axvline(CONFIG["published_auc_vs_calm"], c="k", ls="--", lw=.8, label="published 0.996")
ax[0].axvline(0.5, c="k", ls=":", lw=.8, label="chance")
ax[0].set_xlim(0.4, 1.0); ax[0].set_xlabel("ROC AUC (out-of-fold)")
ax[0].set_title("red: VIVAE vs CREMA-D per padding condition\ngray: padding type, same clips")
ax[0].legend(fontsize=8, loc="lower right")
for corpus, c in (("VIVAE", "tab:red"), ("CREMA-D", "tab:blue")):
    sub = clips[clips.corpus == corpus]
    ax[1].scatter(sub.duration_s, sub.gate_zeros.clip(1e-8), s=10, alpha=.6, c=c, label=f"{corpus}, zero pad")
    ax[1].scatter(sub.duration_s, sub.gate_none.clip(1e-8), s=10, alpha=.6, c=c, marker="x", label=f"{corpus}, no pad")
ax[1].axhline(RULES["DISTRESS_HIGH"], c="k", ls="--", lw=.8, label="DISTRESS_HIGH")
ax[1].axvline(WIN / SR, c="gray", ls=":", lw=.8)
ax[1].set_yscale("log"); ax[1].set_xlabel("clip duration (s)"); ax[1].set_ylabel("current distress-gate score")
ax[1].set_title("current gate score vs duration"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT / "padding_diagnostic.png", dpi=150); plt.show()

result = {
    "seed": CONFIG["seed"], "n_per_corpus": clips.corpus.value_counts().to_dict(),
    "vivae_files": len(vd_all), "vivae_unique_by_md5": int(vd_all.md5.nunique()),
    "level": "src.preprocess.load_audio (-23 LUFS)", "window_s": WIN / SR,
    "feat_extract_norm": wavlm.config.feat_extract_norm,
    "padding_type_auc": pad_auc,
    "vivae_vs_crema_auc": corpus_auc,
    "published_auc_vs_calm": CONFIG["published_auc_vs_calm"],
    "duration_summary": clips.groupby("corpus").duration_s.describe().round(3).to_dict("index"),
    "duration_vs_gate": dur_tbl.to_dict("records"),
}
json.dump(result, open(OUT / "padding_diagnostic.json", "w"), indent=2, default=float)
print(json.dumps({k: result[k] for k in ("padding_type_auc", "vivae_vs_crema_auc")}, indent=2))

## Sections 2–7

Not written yet, deliberately. Spec: do not re-extract anything until Section 1 has
been reviewed. Its numbers decide how much of the published 0.996 survives and therefore
what the refit has to show.